In [3]:
!pip install -q duckdb polars pandas numpy

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/dataset")

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

print("\nTrain files:")
for f in TRAIN_DIR.iterdir():
    print(" -", f.name)

print("\nTest files:")
for f in TEST_DIR.iterdir():
    print(" -", f.name)

Train directory: /content/drive/MyDrive/dataset/train
Test directory: /content/drive/MyDrive/dataset/test

Train files:
 - train_ground_truth.tsv
 - train_source1.tsv
 - train_source2.tsv
 - train_source3.tsv

Test files:
 - test_source1.tsv
 - test_source2.tsv
 - test_source3.tsv


In [6]:
import duckdb

con = duckdb.connect()

files = {
    "source1": TRAIN_DIR / "train_source1.tsv",
    "source2": TRAIN_DIR / "train_source2.tsv",
    "source3": TRAIN_DIR / "train_source3.tsv",
    "ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
}

for name, path in files.items():
    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    # Show column names and types
    schema = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_csv(
            '{path}',
            delim='\\t',
            header=true,
            auto_detect=true
        )
    """).fetchall()

    print("Columns:")
    for row in schema:
        print(f"  {row[0]} -> {row[1]}")

    # Count rows
    count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv(
            '{path}',
            delim='\\t',
            header=true,
            auto_detect=true
        )
    """).fetchone()[0]

    print(f"\nRows: {count:,}")


SOURCE1
Columns:
  entity_id -> VARCHAR
  business_name -> VARCHAR
  business_address -> VARCHAR
  country -> VARCHAR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows: 2,206,821

SOURCE2
Columns:
  entity_id -> VARCHAR
  business_name -> VARCHAR
  business_address -> VARCHAR
  country -> VARCHAR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows: 5,034,616

SOURCE3
Columns:
  entity_id -> VARCHAR
  business_name -> VARCHAR
  business_address -> VARCHAR
  country -> VARCHAR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows: 5,285,603

GROUND_TRUTH
Columns:
  source1_entity_id -> VARCHAR
  matched_entity_ids -> VARCHAR


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows: 2,206,821


In [7]:
# Inspect a small sample of the ground truth

gt_path = TRAIN_DIR / "train_ground_truth.tsv"

sample_gt = con.execute(f"""
    SELECT *
    FROM read_csv(
        '{gt_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    LIMIT 20
""").df()

display(sample_gt)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [8]:
gt_path = TRAIN_DIR / "train_ground_truth.tsv"

match_stats = con.execute(f"""
WITH gt AS (
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_csv(
        '{gt_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),
counts AS (
    SELECT
        source1_entity_id,
        CASE
            WHEN matched_entity_ids IS NULL
                 OR TRIM(matched_entity_ids) = ''
            THEN 0
            ELSE ARRAY_LENGTH(
                STRING_SPLIT(matched_entity_ids, ',')
            )
        END AS match_count
    FROM gt
)
SELECT
    COUNT(*) AS total_source1,
    SUM(CASE WHEN match_count = 0 THEN 1 ELSE 0 END) AS zero_matches,
    SUM(CASE WHEN match_count = 1 THEN 1 ELSE 0 END) AS singleton_count,
    AVG(match_count) AS average_matches,
    MEDIAN(match_count) AS p50,
    QUANTILE_CONT(match_count, 0.90) AS p90,
    QUANTILE_CONT(match_count, 0.95) AS p95,
    QUANTILE_CONT(match_count, 0.99) AS p99,
    MAX(match_count) AS maximum_matches
FROM counts
""").df()

display(match_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_source1,zero_matches,singleton_count,average_matches,p50,p90,p95,p99,maximum_matches
0,2206821,123247.0,119157.0,3.461253,3.0,6.0,6.0,8.0,11


In [9]:
singleton_pct = (
    match_stats.loc[0, "singleton_count"]
    / match_stats.loc[0, "total_source1"]
) * 100

print(f"Singleton %: {singleton_pct:.4f}%")

Singleton %: 5.3995%


In [10]:
s2_s3_stats = con.execute(f"""
WITH gt AS (
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_csv(
        '{gt_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),
matches AS (
    SELECT
        source1_entity_id,
        TRIM(match_id) AS match_id
    FROM gt,
    UNNEST(
        CASE
            WHEN matched_entity_ids IS NULL
                 OR TRIM(matched_entity_ids) = ''
            THEN []
            ELSE STRING_SPLIT(matched_entity_ids, ',')
        END
    ) AS t(match_id)
)
SELECT
    SUM(CASE WHEN match_id LIKE 'S2-%' THEN 1 ELSE 0 END) AS s2_matches,
    SUM(CASE WHEN match_id LIKE 'S3-%' THEN 1 ELSE 0 END) AS s3_matches,
    COUNT(*) AS total_matches
FROM matches
""").df()

display(s2_s3_stats)

total = s2_s3_stats.loc[0, "total_matches"]
s2 = s2_s3_stats.loc[0, "s2_matches"]
s3 = s2_s3_stats.loc[0, "s3_matches"]

print(f"S2 percentage: {(s2 / total) * 100:.2f}%")
print(f"S3 percentage: {(s3 / total) * 100:.2f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,s2_matches,s3_matches,total_matches
0,3693619.0,3944746.0,7638365


S2 percentage: 48.36%
S3 percentage: 51.64%


In [11]:
# Country analysis of ground-truth matches

s1_country_stats = con.execute(f"""
SELECT
    country,
    COUNT(*) AS source1_count
FROM read_csv(
    '{TRAIN_DIR / "train_source1.tsv"}',
    delim='\t',
    header=true,
    auto_detect=true
)
GROUP BY country
ORDER BY source1_count DESC
""").df()

display(s1_country_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country,source1_count
0,US,1323633
1,India,883188


In [12]:
# Check whether true matches have the same country

s1_path = TRAIN_DIR / "train_source1.tsv"
s2_path = TRAIN_DIR / "train_source2.tsv"
s3_path = TRAIN_DIR / "train_source3.tsv"
gt_path = TRAIN_DIR / "train_ground_truth.tsv"

cross_country_stats = con.execute(f"""
WITH s1 AS (
    SELECT
        entity_id AS s1_id,
        country AS s1_country
    FROM read_csv(
        '{s1_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

s2 AS (
    SELECT
        entity_id AS match_id,
        country AS match_country
    FROM read_csv(
        '{s2_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

s3 AS (
    SELECT
        entity_id AS match_id,
        country AS match_country
    FROM read_csv(
        '{s3_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

all_matches AS (
    SELECT * FROM s2
    UNION ALL
    SELECT * FROM s3
),

gt_matches AS (
    SELECT
        source1_entity_id AS s1_id,
        TRIM(match_id) AS match_id
    FROM read_csv(
        '{gt_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    ),
    UNNEST(
        CASE
            WHEN matched_entity_ids IS NULL
                 OR TRIM(matched_entity_ids) = ''
            THEN []
            ELSE STRING_SPLIT(matched_entity_ids, ',')
        END
    ) AS t(match_id)
)

SELECT
    COUNT(*) AS total_matches,
    SUM(
        CASE
            WHEN s1.s1_country = m.match_country THEN 1
            ELSE 0
        END
    ) AS same_country_matches,
    SUM(
        CASE
            WHEN s1.s1_country != m.match_country THEN 1
            ELSE 0
        END
    ) AS cross_country_matches
FROM gt_matches g
JOIN s1
    ON g.s1_id = s1.s1_id
JOIN all_matches m
    ON g.match_id = m.match_id
""").df()

display(cross_country_stats)

total = cross_country_stats.loc[0, "total_matches"]
same = cross_country_stats.loc[0, "same_country_matches"]
cross = cross_country_stats.loc[0, "cross_country_matches"]

print(f"Same-country matches:  {same:,} ({same/total*100:.4f}%)")
print(f"Cross-country matches: {cross:,} ({cross/total*100:.4f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_matches,same_country_matches,cross_country_matches
0,7638365,7638365.0,0.0


Same-country matches:  7,638,365.0 (100.0000%)
Cross-country matches: 0.0 (0.0000%)


In [13]:
# Check for duplicate / strange ground-truth cases

gt_path = TRAIN_DIR / "train_ground_truth.tsv"

gt_quality = con.execute(f"""
WITH gt AS (
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_csv(
        '{gt_path}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

matches AS (
    SELECT
        source1_entity_id,
        TRIM(match_id) AS match_id
    FROM gt,
    UNNEST(
        CASE
            WHEN matched_entity_ids IS NULL
                 OR TRIM(matched_entity_ids) = ''
            THEN []
            ELSE STRING_SPLIT(matched_entity_ids, ',')
        END
    ) AS t(match_id)
),

duplicate_matches AS (
    SELECT
        source1_entity_id,
        match_id,
        COUNT(*) AS occurrences
    FROM matches
    GROUP BY source1_entity_id, match_id
    HAVING COUNT(*) > 1
),

invalid_prefixes AS (
    SELECT DISTINCT
        match_id
    FROM matches
    WHERE NOT (
        match_id LIKE 'S2-%'
        OR match_id LIKE 'S3-%'
    )
)

SELECT
    (SELECT COUNT(*) FROM gt) AS gt_rows,
    (SELECT COUNT(DISTINCT source1_entity_id) FROM gt) AS unique_s1_ids,
    (SELECT COUNT(*) FROM duplicate_matches) AS duplicate_s1_match_pairs,
    (SELECT COUNT(*) FROM invalid_prefixes) AS invalid_match_ids
""").df()

display(gt_quality)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gt_rows,unique_s1_ids,duplicate_s1_match_pairs,invalid_match_ids
0,2206821,2206821,0,0


In [14]:
import duckdb

sample_names = con.execute(f"""
SELECT
    entity_id,
    business_name,
    country
FROM read_csv(
    '{TRAIN_DIR / "train_source1.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND TRIM(business_name) != ''
LIMIT 30
""").df()

display(sample_names)

,entity_id,business_name,country
0,S1-925783039,Orelee's Barbershop,US
1,S1-773889195,Prime Money,US
2,S1-377745466,B+ Retail Inc,US
3,S1-133037285,Christ Chapel,US
4,S1-755362802,Prabhav Business Center,India
5,S1-851869949,Custom Wealth Services LLC,US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,India
7,S1-27541239,Nexus Anchor Rain,US
8,S1-629417405,Moore Bitwise Inc,US
9,S1-22305073,Dermatology Green Medicine,US


In [15]:
import re
import unicodedata

def normalize_name_basic(name):
    if name is None:
        return ""

    name = str(name).lower()

    # Normalize Unicode characters
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )

    # Replace & with "and"
    name = name.replace("&", " and ")

    # Remove punctuation
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name).strip()

    return name


sample_names["normalized_name"] = sample_names["business_name"].apply(
    normalize_name_basic
)

display(
    sample_names[
        ["business_name", "normalized_name", "country"]
    ]
)

,business_name,normalized_name,country
0,Orelee's Barbershop,orelee s barbershop,US
1,Prime Money,prime money,US
2,B+ Retail Inc,b retail inc,US
3,Christ Chapel,christ chapel,US
4,Prabhav Business Center,prabhav business center,India
5,Custom Wealth Services LLC,custom wealth services llc,US
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,India
7,Nexus Anchor Rain,nexus anchor rain,US
8,Moore Bitwise Inc,moore bitwise inc,US
9,Dermatology Green Medicine,dermatology green medicine,US


In [16]:
# Check common legal/business suffixes

suffix_stats = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        regexp_matches(
            lower(business_name),
            '\\b(inc|incorporated|llc|ltd|limited|corp|corporation|company|co|pc|plc)\\b'
        )
    ) AS rows_with_legal_suffix
FROM read_csv(
    '{TRAIN_DIR / "train_source1.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
""").df()

display(suffix_stats)

total = suffix_stats.loc[0, "total_rows"]
with_suffix = suffix_stats.loc[0, "rows_with_legal_suffix"]

print(f"Rows with common legal suffix: {with_suffix:,}")
print(f"Percentage: {with_suffix / total * 100:.2f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_legal_suffix
0,2206821,1377657.0


Rows with common legal suffix: 1,377,657.0
Percentage: 62.43%


In [17]:
suffix_counts = con.execute(f"""
WITH base AS (
    SELECT
        lower(trim(business_name)) AS name
    FROM read_csv(
        '{TRAIN_DIR / "train_source1.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    WHERE business_name IS NOT NULL
)
SELECT
    suffix,
    COUNT(*) AS count
FROM base,
UNNEST([
    'inc',
    'incorporated',
    'llc',
    'ltd',
    'limited',
    'corp',
    'corporation',
    'company',
    'co',
    'pc',
    'plc',
    'private limited'
]) AS t(suffix)
WHERE regexp_matches(name, '\\b' || suffix || '\\b')
GROUP BY suffix
ORDER BY count DESC
""").df()

display(suffix_counts)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,suffix,count
0,limited,522340
1,private limited,431872
2,llc,355736
3,inc,238309
4,ltd,148598
5,corp,34738
6,co,27544
7,pc,27084
8,corporation,14372
9,company,13872


In [19]:
import re
import unicodedata

LEGAL_SUFFIXES = [
    "private limited",
    "incorporated",
    "corporation",
    "company",
    "limited",
    "llc",
    "inc",
    "ltd",
    "corp",
    "co",
    "pc",
    "plc",
]

def normalize_name(name):
    if name is None:
        return "", ""

    name = str(name).lower().strip()

    # Unicode normalization
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )

    # Normalize ampersand
    name = name.replace("&", " and ")

    # Replace punctuation with spaces
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name).strip()

    normalized = name

    # Remove legal suffix from the END only
    core = normalized

    for suffix in sorted(LEGAL_SUFFIXES, key=len, reverse=True):
        pattern = rf"\b{re.escape(suffix)}$"
        core = re.sub(pattern, "", core).strip()

    return normalized, core


# Test on our real sample
sample_names[["name_normalized", "name_core"]] = (
    sample_names["business_name"]
    .apply(lambda x: normalize_name(x))
    .apply(lambda x: __import__("pandas").Series(x))
)

display(
    sample_names[
        ["business_name", "name_normalized", "name_core", "country"]
    ]
)

,business_name,name_normalized,name_core,country
0,Orelee's Barbershop,orelee s barbershop,orelee s barbershop,US
1,Prime Money,prime money,prime money,US
2,B+ Retail Inc,b retail inc,b retail,US
3,Christ Chapel,christ chapel,christ chapel,US
4,Prabhav Business Center,prabhav business center,prabhav business center,India
5,Custom Wealth Services LLC,custom wealth services llc,custom wealth services,US
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,consulting nyasa nursing,India
7,Nexus Anchor Rain,nexus anchor rain,nexus anchor rain,US
8,Moore Bitwise Inc,moore bitwise inc,moore bitwise,US
9,Dermatology Green Medicine,dermatology green medicine,dermatology green medicine,US


In [20]:
# Inspect real ground-truth name pairs

query = f"""
WITH gt AS (
    SELECT
        source1_entity_id,
        TRIM(match_id) AS match_id
    FROM read_csv(
        '{TRAIN_DIR / "train_ground_truth.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    ),
    UNNEST(
        CASE
            WHEN matched_entity_ids IS NULL
                 OR TRIM(matched_entity_ids) = ''
            THEN []
            ELSE STRING_SPLIT(matched_entity_ids, ',')
        END
    ) AS t(match_id)
    LIMIT 30
),

s1 AS (
    SELECT
        entity_id,
        business_name,
        country
    FROM read_csv(
        '{TRAIN_DIR / "train_source1.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

s2 AS (
    SELECT
        entity_id,
        business_name,
        country
    FROM read_csv(
        '{TRAIN_DIR / "train_source2.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

s3 AS (
    SELECT
        entity_id,
        business_name,
        country
    FROM read_csv(
        '{TRAIN_DIR / "train_source3.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

all_sources AS (
    SELECT * FROM s2
    UNION ALL
    SELECT * FROM s3
)

SELECT
    gt.source1_entity_id,
    s1.business_name AS s1_name,
    all_sources.entity_id AS matched_id,
    all_sources.business_name AS matched_name,
    s1.country
FROM gt
JOIN s1
    ON gt.source1_entity_id = s1.entity_id
JOIN all_sources
    ON gt.match_id = all_sources.entity_id
LIMIT 30
"""

true_pairs = con.execute(query).df()

true_pairs["s1_normalized"] = true_pairs["s1_name"].apply(
    lambda x: normalize_name(x)[0]
)

true_pairs["match_normalized"] = true_pairs["matched_name"].apply(
    lambda x: normalize_name(x)[0]
)

true_pairs["s1_core"] = true_pairs["s1_name"].apply(
    lambda x: normalize_name(x)[1]
)

true_pairs["match_core"] = true_pairs["matched_name"].apply(
    lambda x: normalize_name(x)[1]
)

display(true_pairs)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source1_entity_id,s1_name,matched_id,matched_name,country,s1_normalized,match_normalized,s1_core,match_core
0,S1-211827413,Urbandale Diamond Crest,S2-79406660,Urbandale Dead Crest,US,urbandale diamond crest,urbandale dead crest,urbandale diamond crest,urbandale dead crest
1,S1-420503691,Family Physicians,S2-505586486,Family Physicians,US,family physicians,family physicians,family physicians,family physicians
2,S1-211827413,Urbandale Diamond Crest,S2-925080206,Urbandale-Diamond Crest,US,urbandale diamond crest,urbandale diamond crest,urbandale diamond crest,urbandale diamond crest
3,S1-789555951,Iveneon Cap L.L.C.,S2-754199710,Iveneon Cap,US,iveneon cap l l c,iveneon cap,iveneon cap l l c,iveneon cap
4,S1-85939751,Verena's Lending,S2-190847807,Verena's-(Lending),US,verena s lending,verena s lending,verena s lending,verena s lending
5,S1-873267055,Vais PLLC,S2-23280111,Vais PLLC,US,vais pllc,vais pllc,vais pllc,vais pllc
6,S1-85939751,Verena's Lending,S2-414041380,Verena's Lending,US,verena s lending,verena s lending,verena s lending,verena s lending
7,S1-631313902,Aguilar Allied Black LLC,S2-473713811,Aguilar Allied Black LLC,US,aguilar allied black llc,aguilar allied black llc,aguilar allied black,aguilar allied black
8,S1-638981683,GZ Solarius LLC,S2-922247107,gzsolarius.com,US,gz solarius llc,gzsolarius com,gz solarius,gzsolarius com
9,S1-830709217,Great Hospitality Private Limited,S2-250462738,ಗ್ರೇಟ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,India,great hospitality private limited,,great hospitality,


In [21]:
!pip -q install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 1.6 MB/s eta 0:00:00


In [22]:
from unidecode import unidecode

test_unicode = [
    "Great Hospitality Private Limited",
    "ಗ್ರೇಟ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್",
    "Verena's Lénding",
    "Ivene0n Cap"
]

for name in test_unicode:
    print(f"Original     : {name}")
    print(f"Transliterated: {unidecode(name)}")
    print()

Original     : Great Hospitality Private Limited
Transliterated: Great Hospitality Private Limited

Original     : ಗ್ರೇಟ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್
Transliterated: greett haaspittaalitti praiveett limittedd

Original     : Verena's Lénding
Transliterated: Verena's Lending

Original     : Ivene0n Cap
Transliterated: Ivene0n Cap



In [27]:
import re
import unicodedata
from unidecode import unidecode

LEGAL_SUFFIXES = [
    "private limited",
    "incorporated",
    "corporation",
    "company",
    "limited",
    "llc",
    "ltd",
    "inc",
    "corp",
    "co",
    "pc",
    "plc",
    "pllc",
]

def normalize_name_v2(name):
    if name is None:
        return "", ""

    # Convert to string
    name = str(name).strip()

    # Unicode normalization
    name = unicodedata.normalize("NFKD", name)

    # Transliterate non-Latin scripts
    name = unidecode(name)

    # Lowercase
    name = name.lower()

    # & -> and
    name = name.replace("&", " and ")

    # Replace punctuation with spaces
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name).strip()

    normalized = name

    # Remove trailing legal suffix
    core = normalized

    for suffix in sorted(LEGAL_SUFFIXES, key=len, reverse=True):
        pattern = rf"\b{re.escape(suffix)}$"
        core = re.sub(pattern, "", core).strip()

    return normalized, core


# Test on the actual difficult examples
test_names = [
    "Great Hospitality Private Limited",
    "ಗ್ರೇಟ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್",
    "Iveneon Cap L.L.C.",
    "Iveneon Cap",
    "Verena's Lénding",
    "GZ Solarius LLC",
    "Orbidovaaria Co DBA: GZ Solarius LLC",
    "Ivene0n Cap"
]

for name in test_names:
    normalized, core = normalize_name_v2(name)

    print(f"Original   : {name}")
    print(f"Normalized : {normalized}")
    print(f"Core       : {core}")
    print("-" * 70)

Original   : Great Hospitality Private Limited
Normalized : great hospitality private limited
Core       : great hospitality
----------------------------------------------------------------------
Original   : ಗ್ರೇಟ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್
Normalized : gre tt haaspittaalitti pre ve tt limittedd
Core       : gre tt haaspittaalitti pre ve tt limittedd
----------------------------------------------------------------------
Original   : Iveneon Cap L.L.C.
Normalized : iveneon cap l l c
Core       : iveneon cap l l c
----------------------------------------------------------------------
Original   : Iveneon Cap
Normalized : iveneon cap
Core       : iveneon cap
----------------------------------------------------------------------
Original   : Verena's Lénding
Normalized : verena s lending
Core       : verena s lending
----------------------------------------------------------------------
Original   : GZ Solarius LLC
Normalized : gz solarius llc
Core       : gz solarius
---------------

In [29]:
import re
import unicodedata
from unidecode import unidecode

LEGAL_SUFFIXES = [
    "private limited",
    "incorporated",
    "corporation",
    "company",
    "limited",
    "llc",
    "ltd",
    "inc",
    "corp",
    "co",
    "pc",
    "plc",
    "pllc",
]

def normalize_name_v3(name):
    if name is None:
        return "", ""

    name = str(name).strip()

    # Unicode normalization
    name = unicodedata.normalize("NFKD", name)

    # Transliterate non-Latin scripts
    name = unidecode(name)

    # Lowercase
    name = name.lower()

    # Handle dotted abbreviations BEFORE removing punctuation
    name = re.sub(r"\bl\s*\.?\s*l\s*\.?\s*c\.?\b", "llc", name)
    name = re.sub(r"\bp\s*\.?\s*l\s*\.?\s*l\s*\.?\s*c\.?\b", "pllc", name)
    name = re.sub(r"\bi\s*\.?\s*n\s*\.?\s*c\.?\b", "inc", name)
    name = re.sub(r"\bl\s*\.?\s*t\s*\.?\s*d\.?\b", "ltd", name)
    name = re.sub(r"\bp\s*\.?\s*c\.?\b", "pc", name)

    # & → and
    name = name.replace("&", " and ")

    # Remove remaining punctuation
    name = re.sub(r"[^a-z0-9\s]", " ", name)

    # Normalize spaces
    name = re.sub(r"\s+", " ", name).strip()

    normalized = name

    # Remove only the trailing legal suffix
    core = normalized

    for suffix in sorted(LEGAL_SUFFIXES, key=len, reverse=True):
        pattern = rf"\b{re.escape(suffix)}$"

        if re.search(pattern, core):
            core = re.sub(pattern, "", core).strip()
            break

    return normalized, core

In [30]:
test_names = [
    "Iveneon Cap L.L.C.",
    "Iveneon Cap",
    "Vais PLLC",
    "Kelly Advisory, Inc",
    "Dream Construction Limited",
    "Great Hospitality Private Limited"
]

for name in test_names:
    normalized, core = normalize_name_v3(name)

    print(f"Original   : {name}")
    print(f"Normalized : {normalized}")
    print(f"Core       : {core}")
    print("-" * 60)

Original   : Iveneon Cap L.L.C.
Normalized : iveneon cap llc
Core       : iveneon cap
------------------------------------------------------------
Original   : Iveneon Cap
Normalized : iveneon cap
Core       : iveneon cap
------------------------------------------------------------
Original   : Vais PLLC
Normalized : vais pllc
Core       : vais
------------------------------------------------------------
Original   : Kelly Advisory, Inc
Normalized : kelly advisory inc
Core       : kelly advisory
------------------------------------------------------------
Original   : Dream Construction Limited
Normalized : dream construction limited
Core       : dream construction
------------------------------------------------------------
Original   : Great Hospitality Private Limited
Normalized : great hospitality private limited
Core       : great hospitality
------------------------------------------------------------


In [31]:

!pip -q install rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.9 MB/s eta 0:00:00


In [32]:
from rapidfuzz.fuzz import ratio, token_set_ratio, token_sort_ratio

def name_similarity(name1, name2):
    n1, c1 = normalize_name_v3(name1)
    n2, c2 = normalize_name_v3(name2)

    return {
        "normalized_ratio": ratio(n1, n2),
        "core_ratio": ratio(c1, c2),
        "normalized_token_set": token_set_ratio(n1, n2),
        "core_token_set": token_set_ratio(c1, c2),
        "normalized_token_sort": token_sort_ratio(n1, n2),
        "core_token_sort": token_sort_ratio(c1, c2),
    }

In [33]:
results = []

for _, row in true_pairs.iterrows():
    scores = name_similarity(
        row["s1_name"],
        row["matched_name"]
    )

    results.append({
        "s1_name": row["s1_name"],
        "matched_name": row["matched_name"],
        **scores
    })

name_similarity_df = __import__("pandas").DataFrame(results)

display(name_similarity_df)

,s1_name,matched_name,normalized_ratio,core_ratio,normalized_token_set,core_token_set,normalized_token_sort,core_token_sort
0,Urbandale Diamond Crest,Urbandale Dead Crest,88.372093,88.372093,88.372093,88.372093,88.372093,88.372093
1,Family Physicians,Family Physicians,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
2,Urbandale Diamond Crest,Urbandale-Diamond Crest,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
3,Iveneon Cap L.L.C.,Iveneon Cap,84.615385,100.000000,100.000000,100.000000,84.615385,100.000000
4,Verena's Lending,Verena's-(Lending),100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
5,Vais PLLC,Vais PLLC,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
6,Verena's Lending,Verena's Lending,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
7,Aguilar Allied Black LLC,Aguilar Allied Black LLC,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
8,GZ Solarius LLC,gzsolarius.com,82.758621,80.000000,68.965517,80.000000,68.965517,80.000000
9,Great Hospitality Private Limited,ಗ್ರೇಟ್ ಹಾಸ್ಪಿಟಾಲಿಟಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,72.000000,47.457627,72.222222,46.428571,69.333333,44.067797


In [34]:
# Sample 10,000 real ground-truth matches

true_pairs_10k = con.execute(f"""
WITH gt AS (
    SELECT
        source1_entity_id,
        TRIM(match_id) AS match_id
    FROM read_csv(
        '{TRAIN_DIR / "train_ground_truth.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    ),
    UNNEST(
        CASE
            WHEN matched_entity_ids IS NULL
                 OR TRIM(matched_entity_ids) = ''
            THEN []
            ELSE STRING_SPLIT(matched_entity_ids, ',')
        END
    ) AS t(match_id)
),

s1 AS (
    SELECT entity_id, business_name
    FROM read_csv(
        '{TRAIN_DIR / "train_source1.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

s2 AS (
    SELECT entity_id, business_name
    FROM read_csv(
        '{TRAIN_DIR / "train_source2.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

s3 AS (
    SELECT entity_id, business_name
    FROM read_csv(
        '{TRAIN_DIR / "train_source3.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
),

all_sources AS (
    SELECT * FROM s2
    UNION ALL
    SELECT * FROM s3
)

SELECT
    gt.source1_entity_id,
    s1.business_name AS s1_name,
    all_sources.business_name AS matched_name
FROM gt
JOIN s1
    ON gt.source1_entity_id = s1.entity_id
JOIN all_sources
    ON gt.match_id = all_sources.entity_id
USING SAMPLE 10000
""").df()

print(f"Sampled pairs: {len(true_pairs_10k):,}")
display(true_pairs_10k.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sampled pairs: 10,000


,source1_entity_id,s1_name,matched_name
0,S1-672248851,Mad & Co,Mad &
1,S1-88269445,Amherst Precision Connection,AMHERST PRECISION-CONNECITOON
2,S1-272320428,Metro It Limited,Metro It Limited
3,S1-80320327,Om Marketing Private Limited,M/s Om Marketing Private Limited
4,S1-496183853,Rhodes Runway,RHODES RUNWAY
5,S1-934062151,Optimal Studios PC,Optimal Studios PC
6,S1-821964911,Jain Investment Private Limited,Jain Investment Pirivate Limited
7,S1-484222821,Patriot Safe Solana Group,PATRIOT SAFE SOLANA GROUP
8,S1-543366561,North Care Private Limited,નોર્થ કેર પ્રાઇવેટ લિમિટેડ
9,S1-252628936,Behavioral Health Bright Care Associates,Behavioral Health Bright Care Associates Trust


In [35]:
from rapidfuzz.fuzz import (
    ratio,
    token_set_ratio,
    token_sort_ratio
)
import pandas as pd


def calculate_name_scores(row):
    s1_norm, s1_core = normalize_name_v3(row["s1_name"])
    m_norm, m_core = normalize_name_v3(row["matched_name"])

    return pd.Series({
        "normalized_ratio": ratio(s1_norm, m_norm),
        "core_ratio": ratio(s1_core, m_core),
        "normalized_token_set": token_set_ratio(s1_norm, m_norm),
        "core_token_set": token_set_ratio(s1_core, m_core),
        "normalized_token_sort": token_sort_ratio(s1_norm, m_norm),
        "core_token_sort": token_sort_ratio(s1_core, m_core),
    })


# Calculate scores
similarity_scores = true_pairs_10k.apply(
    calculate_name_scores,
    axis=1
)

# Combine with original data
similarity_data = pd.concat(
    [true_pairs_10k.reset_index(drop=True),
     similarity_scores.reset_index(drop=True)],
    axis=1
)

display(similarity_data.head(10))

,source1_entity_id,s1_name,matched_name,normalized_ratio,core_ratio,normalized_token_set,core_token_set,normalized_token_sort,core_token_sort
0,S1-672248851,Mad & Co,Mad &,82.352941,100.000000,100.000000,100.000000,82.352941,100.000000
1,S1-88269445,Amherst Precision Connection,AMHERST PRECISION-CONNECITOON,94.736842,94.736842,94.736842,94.736842,94.736842,94.736842
2,S1-272320428,Metro It Limited,Metro It Limited,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
3,S1-80320327,Om Marketing Private Limited,M/s Om Marketing Private Limited,93.333333,85.714286,100.000000,100.000000,93.333333,85.714286
4,S1-496183853,Rhodes Runway,RHODES RUNWAY,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
5,S1-934062151,Optimal Studios PC,Optimal Studios PC,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
6,S1-821964911,Jain Investment Private Limited,Jain Investment Pirivate Limited,98.412698,76.923077,98.412698,100.000000,98.412698,76.923077
7,S1-484222821,Patriot Safe Solana Group,PATRIOT SAFE SOLANA GROUP,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
8,S1-543366561,North Care Private Limited,નોર્થ કેર પ્રાઇવેટ લિમિટેડ,76.363636,41.025641,76.363636,66.666667,76.363636,41.025641
9,S1-252628936,Behavioral Health Bright Care Associates,Behavioral Health Bright Care Associates Trust,93.023256,93.023256,100.000000,100.000000,93.023256,93.023256


In [36]:
# Distribution statistics for the 10,000 true matches

score_columns = [
    "normalized_ratio",
    "core_ratio",
    "normalized_token_set",
    "core_token_set",
    "normalized_token_sort",
    "core_token_sort"
]

percentiles = [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]

distribution = similarity_data[score_columns].quantile(
    [p / 100 for p in percentiles]
)

distribution.index = [
    "Min", "P1", "P5", "P10", "P25",
    "P50", "P75", "P90", "P95", "P99", "Max"
]

display(distribution.round(2))

,normalized_ratio,core_ratio,normalized_token_set,core_token_set,normalized_token_sort,core_token_sort
Min,5.88,0.00,5.88,0.00,5.88,0.00
P1,21.62,23.53,22.22,22.86,22.22,22.22
P5,54.24,41.18,52.00,41.38,48.28,38.71
P10,63.16,52.38,66.67,54.55,59.46,50.00
P25,74.29,73.33,88.24,88.36,75.68,72.00
P50,88.00,90.48,100.00,100.00,90.32,91.67
P75,100.00,100.00,100.00,100.00,100.00,100.00
P90,100.00,100.00,100.00,100.00,100.00,100.00
P95,100.00,100.00,100.00,100.00,100.00,100.00
P99,100.00,100.00,100.00,100.00,100.00,100.00


In [37]:
# Create 10,000 random negative pairs
# These pairs are NOT present in the ground truth.

negative_pairs = con.execute(f"""
WITH s1 AS (
    SELECT
        entity_id AS s1_id,
        business_name AS s1_name
    FROM read_csv(
        '{TRAIN_DIR / "train_source1.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    USING SAMPLE 10000
),

s2 AS (
    SELECT
        entity_id AS match_id,
        business_name AS matched_name
    FROM read_csv(
        '{TRAIN_DIR / "train_source2.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    USING SAMPLE 10000
)

SELECT
    s1.s1_id AS source1_entity_id,
    s1.s1_name,
    s2.match_id,
    s2.matched_name
FROM s1
CROSS JOIN s2
USING SAMPLE 10000
""").df()

print(f"Negative pairs sampled: {len(negative_pairs):,}")
display(negative_pairs.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Negative pairs sampled: 10,000


,source1_entity_id,s1_name,match_id,matched_name
0,S1-771056852,Gregory and Gregson Inc,S2-549469261,Vioyuma
1,S1-194933903,All All Foods Private Limited,S2-766763459,आदित्य मीडिया एलएलपी
2,S1-285868170,LP Advanced Manulife,S2-409161346,Valley Pyrophyte Reliable
3,S1-695348339,VVR Foods Ltd,S2-769332072,Heritage Pogram III
4,S1-588800673,Karr United Zeo PC,S2-220765038,Gibson Platinum Liquor Incorporated
5,S1-412758957,Worldlink Manufacturing Pvt Ltd,S2-430423091,Sterling Sierra PLLC Services
6,S1-21635765,Morgan Ligand Inc,S2-761873286,LLC Harrington Électronic
7,S1-251870091,Chiropractic Center Inc,S2-351244879,Hrc Consultants L.L.P.
8,S1-269129114,Taylor Institute of Technology LLC,S2-913349851,स्काई इंडस्ट्रीज प्राइवेट लिमिटेड
9,S1-749964885,Johri Farm Noor Nagar Extn. Jamia Nagar (india...,S2-663249977,Private Lazy Traders Industries Limited


In [38]:
# Calculate name similarity for negative pairs

negative_scores = negative_pairs.apply(
    lambda row: pd.Series(
        name_similarity(
            row["s1_name"],
            row["matched_name"]
        )
    ),
    axis=1
)

negative_similarity_data = pd.concat(
    [
        negative_pairs.reset_index(drop=True),
        negative_scores.reset_index(drop=True)
    ],
    axis=1
)

display(negative_similarity_data.head(10))

,source1_entity_id,s1_name,match_id,matched_name,normalized_ratio,core_ratio,normalized_token_set,core_token_set,normalized_token_sort,core_token_sort
0,S1-771056852,Gregory and Gregson Inc,S2-549469261,Vioyuma,20.000000,23.076923,13.333333,15.384615,13.333333,15.384615
1,S1-194933903,All All Foods Private Limited,S2-766763459,आदित्य मीडिया एलएलपी,37.735849,27.027027,40.816327,30.303030,41.509434,32.432432
2,S1-285868170,LP Advanced Manulife,S2-409161346,Valley Pyrophyte Reliable,31.111111,31.111111,31.111111,31.111111,31.111111,31.111111
3,S1-695348339,VVR Foods Ltd,S2-769332072,Heritage Pogram III,25.000000,21.428571,18.750000,14.285714,18.750000,14.285714
4,S1-588800673,Karr United Zeo PC,S2-220765038,Gibson Platinum Liquor Incorporated,26.415094,21.621622,26.415094,16.216216,26.415094,16.216216
5,S1-412758957,Worldlink Manufacturing Pvt Ltd,S2-430423091,Sterling Sierra PLLC Services,30.000000,32.142857,33.333333,32.142857,33.333333,32.142857
6,S1-21635765,Morgan Ligand Inc,S2-761873286,LLC Harrington Électronic,38.095238,31.578947,33.333333,31.578947,33.333333,31.578947
7,S1-251870091,Chiropractic Center Inc,S2-351244879,Hrc Consultants L.L.P.,36.363636,35.000000,38.095238,36.842105,36.363636,35.000000
8,S1-269129114,Taylor Institute of Technology LLC,S2-913349851,स्काई इंडस्ट्रीज प्राइवेट लिमिटेड,33.333333,35.294118,30.555556,32.352941,30.555556,32.352941
9,S1-749964885,Johri Farm Noor Nagar Extn. Jamia Nagar (india...,S2-663249977,Private Lazy Traders Industries Limited,48.484848,37.333333,55.913978,34.782609,44.444444,34.666667


In [39]:
# Distribution statistics for negative pairs

score_columns = [
    "normalized_ratio",
    "core_ratio",
    "normalized_token_set",
    "core_token_set",
    "normalized_token_sort",
    "core_token_sort"
]

negative_distribution = negative_similarity_data[
    score_columns
].quantile(
    [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
)

negative_distribution.index = [
    "Min", "P1", "P5", "P10", "P25",
    "P50", "P75", "P90", "P95", "P99", "Max"
]

display(negative_distribution.round(2))

,normalized_ratio,core_ratio,normalized_token_set,core_token_set,normalized_token_sort,core_token_sort
Min,0.00,0.00,0.00,0.00,0.00,0.00
P1,15.38,12.90,15.38,12.90,15.38,12.90
P5,21.05,18.75,21.05,18.66,21.05,18.60
P10,23.53,21.43,23.73,21.43,23.73,21.43
P25,28.00,25.81,28.00,26.09,28.00,25.93
P50,32.65,30.51,32.73,30.77,32.65,30.77
P75,37.74,35.29,38.10,35.71,37.74,35.56
P90,43.75,40.00,44.44,40.00,43.14,40.00
P95,50.00,42.42,50.00,43.14,47.76,42.86
P99,61.97,49.06,66.67,51.16,59.38,49.06


In [40]:
import duckdb
from pathlib import Path

TRAIN_DIR = Path("/content/drive/MyDrive/dataset/train")

con = duckdb.connect()

# Lightweight SQL normalization for the blocking experiment
# Keeps the operation scalable over millions of rows.
def sql_name_expr(column):
    return f"""
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    lower(trim({column})),
                    '[^a-z0-9]+', ' ', 'g'
                ),
                '\\\\s+', ' ', 'g'
            ),
            '^ | $', '', 'g'
        )
    """

s1_name = sql_name_expr("business_name")
s2_name = sql_name_expr("business_name")

# ---------------------------------------------------------
# 1. Build normalized Source 2 + Source 3 candidate index
# ---------------------------------------------------------

print("Building S2 normalized index...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s2_index AS
SELECT
    entity_id,
    country,
    {s2_name} AS name_normalized
FROM read_csv(
    '{TRAIN_DIR / "train_source2.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND business_name != '';
""")

print("Building S3 normalized index...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s3_index AS
SELECT
    entity_id,
    country,
    {s2_name} AS name_normalized
FROM read_csv(
    '{TRAIN_DIR / "train_source3.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND business_name != '';
""")

print("Indexes created.")

Building S2 normalized index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Building S3 normalized index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Indexes created.


In [41]:
# Build Source 1 normalized names
print("Building Source 1 index...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s1_index AS
SELECT
    entity_id AS s1_id,
    country,
    {s1_name} AS name_normalized
FROM read_csv(
    '{TRAIN_DIR / "train_source1.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND business_name != '';
""")

# Count candidates from exact country + normalized name
candidate_stats = con.execute("""
WITH candidates AS (
    SELECT
        s1.s1_id,
        COUNT(*) AS candidate_count
    FROM s1_index s1
    JOIN s2_index s2
      ON s1.country = s2.country
     AND s1.name_normalized = s2.name_normalized
    GROUP BY s1.s1_id

    UNION ALL

    SELECT
        s1.s1_id,
        COUNT(*) AS candidate_count
    FROM s1_index s1
    JOIN s3_index s3
      ON s1.country = s3.country
     AND s1.name_normalized = s3.name_normalized
    GROUP BY s1.s1_id
)

SELECT
    COUNT(*) AS s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM candidates
""").df()

display(candidate_stats)

Building Source 1 index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,2399040,9.070983,2.0,28.0,46.0,106.0,526


In [42]:
# ============================================================
# Blocking Experiment 1 Evaluation
# Country + Exact Normalized Name
# ============================================================

print("Building exact-name candidate pairs...")

con.execute("""
CREATE OR REPLACE TEMP TABLE exact_candidates AS

SELECT
    s1.s1_id,
    s2.entity_id AS matched_id
FROM s1_index s1
JOIN s2_index s2
  ON s1.country = s2.country
 AND s1.name_normalized = s2.name_normalized

UNION

SELECT
    s1.s1_id,
    s3.entity_id AS matched_id
FROM s1_index s1
JOIN s3_index s3
  ON s1.country = s3.country
 AND s1.name_normalized = s3.name_normalized
""")

print("Candidate pairs created.")

# ------------------------------------------------------------
# Ground truth: explode comma-separated matched IDs
# ------------------------------------------------------------

print("Reading ground truth...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE gt_pairs AS
SELECT
    source1_entity_id AS s1_id,
    TRIM(match_id) AS matched_id
FROM read_csv(
    '{TRAIN_DIR / "train_ground_truth.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
),
UNNEST(string_split(matched_entity_ids, ',')) AS t(match_id)
WHERE matched_entity_ids IS NOT NULL
  AND TRIM(matched_entity_ids) != '';
""")

# ------------------------------------------------------------
# 1. Pair-level candidate recall
# ------------------------------------------------------------

pair_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(ec.matched_id) AS true_pairs_found,
    COUNT(ec.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN exact_candidates ec
    ON gt.s1_id = ec.s1_id
   AND gt.matched_id = ec.matched_id
""").df()

print("\n=== PAIR-LEVEL RECALL ===")
display(pair_recall.round(4))


# ------------------------------------------------------------
# 2. Unique S1 candidate-set statistics
# ------------------------------------------------------------

candidate_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM exact_candidates
    GROUP BY s1_id
)

SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== CANDIDATE SET SIZE ===")
display(candidate_stats.round(4))


# ------------------------------------------------------------
# 3. Entity-level recall
#    How many S1 entities have ALL their true matches
#    present in the candidate set?
# ------------------------------------------------------------

entity_recall = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),

found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(ec.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN exact_candidates ec
        ON gt.s1_id = ec.s1_id
       AND gt.matched_id = ec.matched_id
    GROUP BY gt.s1_id
)

SELECT
    COUNT(*) AS s1_with_true_matches,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct

FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== ENTITY-LEVEL RECALL ===")
display(entity_recall.round(4))

Building exact-name candidate pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate pairs created.
Reading ground truth...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...


=== CANDIDATE SET SIZE ===


,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,1564470,13.9099,2.0,35.0,78.0,199.0,1042


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== ENTITY-LEVEL RECALL ===


,s1_with_true_matches,s1_all_matches_found,all_match_entity_recall_pct
0,2083574,58340.0,2.8


In [43]:
# ============================================================
# Blocking Experiment 2
# Country + Exact Core Name
# ============================================================

print("Building core-name indexes...")

# SQL expression that approximately removes common legal suffixes
# after basic normalization.
def sql_core_name_expr(column):
    normalized = sql_name_expr(column)

    return f"""
        regexp_replace(
            {normalized},
            '\\\\s+(private limited|incorporated|corporation|company|limited|llc|ltd|inc|corp|co|pc|plc|pllc)$',
            '',
            'g'
        )
    """

core_expr = sql_core_name_expr("business_name")


con.execute(f"""
CREATE OR REPLACE TEMP TABLE s2_core_index AS
SELECT
    entity_id,
    country,
    {core_expr} AS name_core
FROM read_csv(
    '{TRAIN_DIR / "train_source2.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND business_name != '';
""")


con.execute(f"""
CREATE OR REPLACE TEMP TABLE s3_core_index AS
SELECT
    entity_id,
    country,
    {core_expr} AS name_core
FROM read_csv(
    '{TRAIN_DIR / "train_source3.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND business_name != '';
""")


con.execute(f"""
CREATE OR REPLACE TEMP TABLE s1_core_index AS
SELECT
    entity_id AS s1_id,
    country,
    {core_expr} AS name_core
FROM read_csv(
    '{TRAIN_DIR / "train_source1.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
)
WHERE business_name IS NOT NULL
  AND business_name != '';
""")

print("Core-name indexes created.")

Building core-name indexes...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Core-name indexes created.


In [44]:
# ============================================================
# Evaluate Experiment 2
# Country + Exact Core Name
# ============================================================

print("Building core-name candidate pairs...")

con.execute("""
CREATE OR REPLACE TEMP TABLE core_candidates AS

SELECT
    s1.s1_id,
    s2.entity_id AS matched_id
FROM s1_core_index s1
JOIN s2_core_index s2
  ON s1.country = s2.country
 AND s1.name_core = s2.name_core
 AND s1.name_core != ''

UNION

SELECT
    s1.s1_id,
    s3.entity_id AS matched_id
FROM s1_core_index s1
JOIN s3_core_index s3
  ON s1.country = s3.country
 AND s1.name_core = s3.name_core
 AND s1.name_core != ''
""")

print("Core-name candidate pairs created.")


# ------------------------------------------------------------
# Pair-level recall
# ------------------------------------------------------------

core_pair_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(ec.matched_id) AS true_pairs_found,
    COUNT(ec.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN core_candidates ec
    ON gt.s1_id = ec.s1_id
   AND gt.matched_id = ec.matched_id
""").df()

print("\n=== CORE-NAME PAIR RECALL ===")
display(core_pair_recall.round(4))


# ------------------------------------------------------------
# Candidate-set size
# ------------------------------------------------------------

core_candidate_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM core_candidates
    GROUP BY s1_id
)

SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== CORE-NAME CANDIDATE SET SIZE ===")
display(core_candidate_stats.round(4))


# ------------------------------------------------------------
# Entity-level recall
# ------------------------------------------------------------

core_entity_recall = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),

found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(ec.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN core_candidates ec
        ON gt.s1_id = ec.s1_id
       AND gt.matched_id = ec.matched_id
    GROUP BY gt.s1_id
)

SELECT
    COUNT(*) AS s1_with_true_matches,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct

FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== CORE-NAME ENTITY RECALL ===")
display(core_entity_recall.round(4))

Building core-name candidate pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Core-name candidate pairs created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== CORE-NAME PAIR RECALL ===


...


=== CORE-NAME CANDIDATE SET SIZE ===


,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,1564470,13.9099,2.0,35.0,78.0,199.0,1042


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== CORE-NAME ENTITY RECALL ===


,s1_with_true_matches,s1_all_matches_found,all_match_entity_recall_pct
0,2083574,58340.0,2.8


In [45]:
# ============================================================
# Blocking Experiment 3
# Country + Shared Name Token
# ============================================================

print("Building token indexes...")

con.execute("""
CREATE OR REPLACE TEMP TABLE s2_tokens AS
SELECT
    entity_id,
    country,
    token
FROM s2_index,
UNNEST(string_split(name_normalized, ' ')) AS t(token)
WHERE token != ''
  AND length(token) >= 3;
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE s3_tokens AS
SELECT
    entity_id,
    country,
    token
FROM s3_index,
UNNEST(string_split(name_normalized, ' ')) AS t(token)
WHERE token != ''
  AND length(token) >= 3;
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE s1_tokens AS
SELECT
    s1_id,
    country,
    token
FROM s1_index,
UNNEST(string_split(name_normalized, ' ')) AS t(token)
WHERE token != ''
  AND length(token) >= 3;
""")

print("Token indexes created.")

Building token indexes...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Token indexes created.


In [ ]:
# ============================================================
# Evaluate Experiment 3
# Country + Shared Name Token
# ============================================================

print("Building token candidate pairs...")

con.execute("""
CREATE OR REPLACE TEMP TABLE token_candidates AS

SELECT DISTINCT
    s1.s1_id,
    s2.entity_id AS matched_id
FROM s1_tokens s1
JOIN s2_tokens s2
  ON s1.country = s2.country
 AND s1.token = s2.token

UNION

SELECT DISTINCT
    s1.s1_id,
    s3.entity_id AS matched_id
FROM s1_tokens s1
JOIN s3_tokens s3
  ON s1.country = s3.country
 AND s1.token = s3.token
""")

print("Token candidate pairs created.")


# ------------------------------------------------------------
# Pair-level recall
# ------------------------------------------------------------

token_pair_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(tc.matched_id) AS true_pairs_found,
    COUNT(tc.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN token_candidates tc
    ON gt.s1_id = tc.s1_id
   AND gt.matched_id = tc.matched_id
""").df()

print("\n=== TOKEN PAIR RECALL ===")
display(token_pair_recall.round(4))


# ------------------------------------------------------------
# Candidate-set size
# ------------------------------------------------------------

token_candidate_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM token_candidates
    GROUP BY s1_id
)

SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== TOKEN CANDIDATE SET SIZE ===")
display(token_candidate_stats.round(4))


# ------------------------------------------------------------
# Entity-level recall
# ------------------------------------------------------------

token_entity_recall = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),

found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(tc.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN token_candidates tc
        ON gt.s1_id = tc.s1_id
       AND gt.matched_id = tc.matched_id
    GROUP BY gt.s1_id
)

SELECT
    COUNT(*) AS s1_with_true_matches,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct

FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== TOKEN ENTITY RECALL ===")
display(token_entity_recall.round(4))

Building token candidate pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [1]:
# ============================================================
# M3 HANDOFF - CURRENT CANDIDATE INTERFACE
# ============================================================

from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/business_entity_resolution/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Basic candidate file
m3_candidates = con.execute("""
SELECT
    s1_id AS source1_entity_id,
    matched_id AS matched_entity_id
FROM exact_candidates
ORDER BY s1_id, matched_id
""").df()

# Save full candidate set
candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"

m3_candidates.to_csv(
    candidate_path,
    sep="\t",
    index=False
)

# Save 500-row sample
sample_path = OUTPUT_DIR / "candidate_pairs_sample_500.tsv"

m3_candidates.head(500).to_csv(
    sample_path,
    sep="\t",
    index=False
)

print("========================================")
print("M3 CANDIDATE HANDOFF")
print("========================================")
print(f"Full candidate file : {candidate_path}")
print(f"Sample file         : {sample_path}")
print(f"Total candidates    : {len(m3_candidates):,}")
print()
print("Columns:")
print(list(m3_candidates.columns))
print()
print("Sample:")
display(m3_candidates.head(10))

NameError: name 'con' is not defined

In [2]:
import duckdb
from pathlib import Path

# Reconnect
con = duckdb.connect()

TRAIN_DIR = Path("/content/drive/MyDrive/dataset/train")
OUTPUT_DIR = Path("/content/drive/MyDrive/business_entity_resolution/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S1 = TRAIN_DIR / "train_source1.tsv"
S2 = TRAIN_DIR / "train_source2.tsv"

print("Creating a 500-row candidate sample...")

# Generate a small sample directly from the dataset.
# No giant candidate table is created.
sample = con.execute(f"""
WITH s1 AS (
    SELECT
        entity_id AS source1_entity_id,
        country,
        lower(trim(business_name)) AS name_normalized
    FROM read_csv(
        '{S1}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    WHERE business_name IS NOT NULL
),

s2 AS (
    SELECT
        entity_id AS matched_entity_id,
        country,
        lower(trim(business_name)) AS name_normalized
    FROM read_csv(
        '{S2}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    WHERE business_name IS NOT NULL
)

SELECT
    s1.source1_entity_id,
    s2.matched_entity_id
FROM s1
JOIN s2
    ON s1.country = s2.country
   AND s1.name_normalized = s2.name_normalized
LIMIT 500
""").df()

sample_path = OUTPUT_DIR / "candidate_pairs_sample_500.tsv"

sample.to_csv(
    sample_path,
    sep="\t",
    index=False
)

print("\n========== M3 HANDOFF ==========")
print("Sample rows:", len(sample))
print("Columns:", list(sample.columns))
print("Sample path:")
print(sample_path)

display(sample.head(10))

Creating a 500-row candidate sample...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


========== M3 HANDOFF ==========
Sample rows: 500
Columns: ['source1_entity_id', 'matched_entity_id']
Sample path:
/content/drive/MyDrive/business_entity_resolution/output/candidate_pairs_sample_500.tsv


,source1_entity_id,matched_entity_id
0,S1-310809132,S2-163963287
1,S1-213064927,S2-868087865
2,S1-434551913,S2-378707035
3,S1-879245051,S2-202057528
4,S1-574186214,S2-235039028
5,S1-14835803,S2-669879950
6,S1-346163560,S2-401480583
7,S1-713558536,S2-208005980
8,S1-179959271,S2-187379771
9,S1-542567244,S2-102309877


In [3]:
# ============================================================
# FAST TOKEN BLOCKING - STEP 1
# Build rare/informative token indexes
# ============================================================

import duckdb
from pathlib import Path

con = duckdb.connect()

TRAIN_DIR = Path("/content/drive/MyDrive/dataset/train")

# Tokens appearing more than this many times are ignored.
MAX_TOKEN_FREQ = 500

print("Building S2 rare-token index...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s2_rare_tokens AS
WITH base AS (
    SELECT
        entity_id,
        country,
        lower(trim(business_name)) AS name_normalized
    FROM read_csv(
        '{TRAIN_DIR / "train_source2.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    WHERE business_name IS NOT NULL
      AND trim(business_name) != ''
),

tokens AS (
    SELECT
        entity_id,
        country,
        token
    FROM base,
    UNNEST(string_split(name_normalized, ' ')) AS t(token)
    WHERE length(token) >= 3
),

token_frequency AS (
    SELECT
        country,
        token,
        COUNT(*) AS freq
    FROM tokens
    GROUP BY country, token
    HAVING COUNT(*) <= {MAX_TOKEN_FREQ}
)

SELECT DISTINCT
    t.entity_id,
    t.country,
    t.token
FROM tokens t
JOIN token_frequency f
  ON t.country = f.country
 AND t.token = f.token;
""")

print("S2 rare-token index created.")

print("Building S3 rare-token index...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s3_rare_tokens AS
WITH base AS (
    SELECT
        entity_id,
        country,
        lower(trim(business_name)) AS name_normalized
    FROM read_csv(
        '{TRAIN_DIR / "train_source3.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    WHERE business_name IS NOT NULL
      AND trim(business_name) != ''
),

tokens AS (
    SELECT
        entity_id,
        country,
        token
    FROM base,
    UNNEST(string_split(name_normalized, ' ')) AS t(token)
    WHERE length(token) >= 3
),

token_frequency AS (
    SELECT
        country,
        token,
        COUNT(*) AS freq
    FROM tokens
    GROUP BY country, token
    HAVING COUNT(*) <= {MAX_TOKEN_FREQ}
)

SELECT DISTINCT
    t.entity_id,
    t.country,
    t.token
FROM tokens t
JOIN token_frequency f
  ON t.country = f.country
 AND t.token = f.token;
""")

print("S3 rare-token index created.")

Building S2 rare-token index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 rare-token index created.
Building S3 rare-token index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S3 rare-token index created.


In [4]:
# ============================================================
# FAST TOKEN BLOCKING - STEP 2
# Build Source 1 rare-token index
# ============================================================

print("Building S1 rare-token index...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE s1_rare_tokens AS
WITH base AS (
    SELECT
        entity_id AS s1_id,
        country,
        lower(trim(business_name)) AS name_normalized
    FROM read_csv(
        '{TRAIN_DIR / "train_source1.tsv"}',
        delim='\\t',
        header=true,
        auto_detect=true
    )
    WHERE business_name IS NOT NULL
      AND trim(business_name) != ''
),

tokens AS (
    SELECT
        s1_id,
        country,
        token
    FROM base,
    UNNEST(string_split(name_normalized, ' ')) AS t(token)
    WHERE length(token) >= 3
),

token_frequency AS (
    SELECT
        country,
        token,
        COUNT(*) AS freq
    FROM tokens
    GROUP BY country, token
    HAVING COUNT(*) <= {MAX_TOKEN_FREQ}
)

SELECT DISTINCT
    t.s1_id,
    t.country,
    t.token
FROM tokens t
JOIN token_frequency f
  ON t.country = f.country
 AND t.token = f.token;
""")

print("S1 rare-token index created.")

Building S1 rare-token index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 rare-token index created.


In [5]:
# ============================================================
# FAST TOKEN BLOCKING - STEP 3
# Generate rare-token candidates
# ============================================================

print("Generating S2 token candidates...")

con.execute("""
CREATE OR REPLACE TEMP TABLE token_candidates AS

SELECT DISTINCT
    s1.s1_id,
    s2.entity_id AS matched_id
FROM s1_rare_tokens s1
JOIN s2_rare_tokens s2
  ON s1.country = s2.country
 AND s1.token = s2.token

UNION

SELECT DISTINCT
    s1.s1_id,
    s3.entity_id AS matched_id
FROM s1_rare_tokens s1
JOIN s3_rare_tokens s3
  ON s1.country = s3.country
 AND s1.token = s3.token
""")

print("Token candidates created.")

Generating S2 token candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Token candidates created.


In [7]:
# ============================================================
# Recreate Ground Truth Pairs
# ============================================================

print("Rebuilding ground-truth pairs...")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE gt_pairs AS
SELECT
    source1_entity_id AS s1_id,
    TRIM(match_id) AS matched_id
FROM read_csv(
    '{TRAIN_DIR / "train_ground_truth.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
),
UNNEST(string_split(matched_entity_ids, ',')) AS t(match_id)
WHERE matched_entity_ids IS NOT NULL
  AND TRIM(matched_entity_ids) != '';
""")

gt_count = con.execute("""
SELECT COUNT(*) AS total_true_pairs
FROM gt_pairs
""").fetchone()[0]

print(f"Ground-truth pairs loaded: {gt_count:,}")

Rebuilding ground-truth pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ground-truth pairs loaded: 7,638,365


In [8]:
# ============================================================
# FAST TOKEN BLOCKING - EVALUATION
# ============================================================

print("Evaluating token blocker...")

# Pair-level recall
token_pair_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(tc.matched_id) AS true_pairs_found,
    COUNT(tc.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN token_candidates tc
    ON gt.s1_id = tc.s1_id
   AND gt.matched_id = tc.matched_id
""").df()

print("\n=== TOKEN PAIR RECALL ===")
display(token_pair_recall.round(4))


# Candidate-set size
token_candidate_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM token_candidates
    GROUP BY s1_id
)

SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== TOKEN CANDIDATE SET SIZE ===")
display(token_candidate_stats.round(4))


# Entity-level recall
token_entity_recall = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),

found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(tc.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN token_candidates tc
        ON gt.s1_id = tc.s1_id
       AND gt.matched_id = tc.matched_id
    GROUP BY gt.s1_id
)

SELECT
    COUNT(*) AS s1_with_true_matches,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,

    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct

FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== TOKEN ENTITY RECALL ===")
display(token_entity_recall.round(4))

Evaluating token blocker...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== TOKEN PAIR RECALL ===


,total_true_pairs,true_pairs_found,pair_recall_pct
0,7638365,3221060,42.1695


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== TOKEN CANDIDATE SET SIZE ===


,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,1101478,258.6442,149.0,701.0,868.0,1150.0,2874


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== TOKEN ENTITY RECALL ===


,s1_with_true_matches,s1_all_matches_found,all_match_entity_recall_pct
0,2083574,601497.0,28.8685


In [2]:
import duckdb
from pathlib import Path

DB_PATH = "/content/drive/MyDrive/business_entity_resolution/blocking.duckdb"

con = duckdb.connect(DB_PATH)

TRAIN_DIR = Path("/content/drive/MyDrive/dataset/train")

print("Persistent DuckDB connected:")
print(DB_PATH)

Persistent DuckDB connected:
/content/drive/MyDrive/business_entity_resolution/blocking.duckdb


In [4]:
# Check persistent DuckDB tables

tables = con.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df()

display(tables)

,table_name


In [5]:
# ============================================================
# PERSIST SOURCE TABLES
# ============================================================

print("Creating persistent Source 1 table...")

con.execute(f"""
CREATE OR REPLACE TABLE s1_data AS
SELECT
    entity_id AS s1_id,
    business_name,
    business_address,
    country,
    lower(trim(business_name)) AS name_normalized
FROM read_csv(
    '{TRAIN_DIR / "train_source1.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
);
""")

print("S1 done.")

print("Creating persistent Source 2 table...")

con.execute(f"""
CREATE OR REPLACE TABLE s2_data AS
SELECT
    entity_id AS matched_id,
    business_name,
    business_address,
    country,
    lower(trim(business_name)) AS name_normalized
FROM read_csv(
    '{TRAIN_DIR / "train_source2.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
);
""")

print("S2 done.")

print("Creating persistent Source 3 table...")

con.execute(f"""
CREATE OR REPLACE TABLE s3_data AS
SELECT
    entity_id AS matched_id,
    business_name,
    business_address,
    country,
    lower(trim(business_name)) AS name_normalized
FROM read_csv(
    '{TRAIN_DIR / "train_source3.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
);
""")

print("S3 done.")

print("================================")
print("All persistent source tables ready")
print("================================")

Creating persistent Source 1 table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 done.
Creating persistent Source 2 table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 done.
Creating persistent Source 3 table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S3 done.
All persistent source tables ready


In [7]:
# ============================================================
# FIX: BUILD S2/S3 RARE TOKEN INDEXES
# ============================================================

print("Building S2 rare-token index...")

con.execute("""
CREATE OR REPLACE TABLE s2_rare_tokens AS
SELECT DISTINCT
    s.matched_id,
    s.country,
    t.token
FROM s2_data s
CROSS JOIN UNNEST(string_split(s.name_normalized, ' ')) AS t(token)
JOIN token_frequency f
  ON s.country = f.country
 AND t.token = f.token
WHERE length(t.token) >= 3
  AND t.token != ''
""")

print("S2 rare-token index created.")

print("Building S3 rare-token index...")

con.execute("""
CREATE OR REPLACE TABLE s3_rare_tokens AS
SELECT DISTINCT
    s.matched_id,
    s.country,
    t.token
FROM s3_data s
CROSS JOIN UNNEST(string_split(s.name_normalized, ' ')) AS t(token)
JOIN token_frequency f
  ON s.country = f.country
 AND t.token = f.token
WHERE length(t.token) >= 3
  AND t.token != ''
""")

print("S3 rare-token index created.")

print("================================")
print("Persistent token indexes ready")
print("================================")

Building S2 rare-token index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 rare-token index created.
Building S3 rare-token index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S3 rare-token index created.
Persistent token indexes ready


In [8]:
# ============================================================
# BUILD PERSISTENT S1 RARE-TOKEN INDEX
# ============================================================

print("Building S1 rare-token index...")

con.execute("""
CREATE OR REPLACE TABLE s1_rare_tokens AS

SELECT DISTINCT
    s.s1_id,
    s.country,
    t.token
FROM s1_data s
CROSS JOIN UNNEST(string_split(s.name_normalized, ' ')) AS t(token)
JOIN token_frequency f
  ON s.country = f.country
 AND t.token = f.token
WHERE length(t.token) >= 3
  AND t.token != ''
""")

print("S1 rare-token index created.")

# Quick size check
counts = con.execute("""
SELECT
    COUNT(*) AS token_rows,
    COUNT(DISTINCT s1_id) AS unique_s1
FROM s1_rare_tokens
""").df()

display(counts)

Building S1 rare-token index...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 rare-token index created.


,token_rows,unique_s1
0,666427,594483


In [9]:
# ============================================================
# BUILD PERSISTENT TWO-TOKEN KEYS
# ============================================================

print("Creating S1 two-token keys...")

con.execute("""
CREATE OR REPLACE TABLE s1_token_pairs AS
SELECT DISTINCT
    a.s1_id,
    a.country,
    a.token AS token_a,
    b.token AS token_b
FROM s1_rare_tokens a
JOIN s1_rare_tokens b
  ON a.s1_id = b.s1_id
 AND a.country = b.country
 AND a.token < b.token
""")

print("S1 two-token keys created.")


print("Creating S2 two-token keys...")

con.execute("""
CREATE OR REPLACE TABLE s2_token_pairs AS
SELECT DISTINCT
    a.matched_id,
    a.country,
    a.token AS token_a,
    b.token AS token_b
FROM s2_rare_tokens a
JOIN s2_rare_tokens b
  ON a.matched_id = b.matched_id
 AND a.country = b.country
 AND a.token < b.token
""")

print("S2 two-token keys created.")


print("Creating S3 two-token keys...")

con.execute("""
CREATE OR REPLACE TABLE s3_token_pairs AS
SELECT DISTINCT
    a.matched_id,
    a.country,
    a.token AS token_a,
    b.token AS token_b
FROM s3_rare_tokens a
JOIN s3_rare_tokens b
  ON a.matched_id = b.matched_id
 AND a.country = b.country
 AND a.token < b.token
""")

print("S3 two-token keys created.")

print("======================================")
print("All two-token indexes are ready")
print("======================================")

Creating S1 two-token keys...
S1 two-token keys created.
Creating S2 two-token keys...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 two-token keys created.
Creating S3 two-token keys...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S3 two-token keys created.
All two-token indexes are ready


In [10]:
# ============================================================
# TWO-TOKEN CANDIDATE GENERATION
# ============================================================

print("Generating S2 two-token candidates...")

con.execute("""
CREATE OR REPLACE TABLE s2_token2_candidates AS
SELECT DISTINCT
    s1.s1_id,
    s2.matched_id
FROM s1_token_pairs s1
JOIN s2_token_pairs s2
  ON s1.country = s2.country
 AND s1.token_a = s2.token_a
 AND s1.token_b = s2.token_b
""")

print("S2 two-token candidates created.")

print("Generating S3 two-token candidates...")

con.execute("""
CREATE OR REPLACE TABLE s3_token2_candidates AS
SELECT DISTINCT
    s1.s1_id,
    s3.matched_id
FROM s1_token_pairs s1
JOIN s3_token_pairs s3
  ON s1.country = s3.country
 AND s1.token_a = s3.token_a
 AND s1.token_b = s3.token_b
""")

print("S3 two-token candidates created.")

print("Combining candidates...")

con.execute("""
CREATE OR REPLACE TABLE token2_candidates AS

SELECT
    s1_id,
    matched_id
FROM s2_token2_candidates

UNION

SELECT
    s1_id,
    matched_id
FROM s3_token2_candidates
""")

print("======================================")
print("Two-token candidate generation done")
print("======================================")

Generating S2 two-token candidates...
S2 two-token candidates created.
Generating S3 two-token candidates...
S3 two-token candidates created.
Combining candidates...
Two-token candidate generation done


In [12]:
# ============================================================
# PERSIST GROUND TRUTH PAIRS
# ============================================================

print("Creating persistent ground-truth table...")

con.execute(f"""
CREATE OR REPLACE TABLE gt_pairs AS
SELECT
    source1_entity_id AS s1_id,
    TRIM(match_id) AS matched_id
FROM read_csv(
    '{TRAIN_DIR / "train_ground_truth.tsv"}',
    delim='\\t',
    header=true,
    auto_detect=true
),
UNNEST(string_split(matched_entity_ids, ',')) AS t(match_id)
WHERE matched_entity_ids IS NOT NULL
  AND TRIM(matched_entity_ids) != '';
""")

gt_count = con.execute("""
SELECT COUNT(*) FROM gt_pairs
""").fetchone()[0]

print(f"Persistent ground-truth pairs: {gt_count:,}")

Creating persistent ground-truth table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Persistent ground-truth pairs: 7,638,365


In [13]:
# ============================================================
# EVALUATE TWO-TOKEN BLOCKER
# ============================================================

print("Evaluating two-token blocker...")

two_token_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(tc.matched_id) AS true_pairs_found,
    COUNT(tc.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN token2_candidates tc
    ON gt.s1_id = tc.s1_id
   AND gt.matched_id = tc.matched_id
""").df()

print("\n=== TWO-TOKEN PAIR RECALL ===")
display(two_token_recall.round(4))


two_token_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM token2_candidates
    GROUP BY s1_id
)

SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== TWO-TOKEN CANDIDATE SET SIZE ===")
display(two_token_stats.round(4))


two_token_entity = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),

found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(tc.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN token2_candidates tc
        ON gt.s1_id = tc.s1_id
       AND gt.matched_id = tc.matched_id
    GROUP BY gt.s1_id
)

SELECT
    COUNT(*) AS s1_with_true_matches,
    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,
    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct
FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== TWO-TOKEN ENTITY RECALL ===")
display(two_token_entity.round(4))

Evaluating two-token blocker...

=== TWO-TOKEN PAIR RECALL ===


,total_true_pairs,true_pairs_found,pair_recall_pct
0,7638365,152388,1.995



=== TWO-TOKEN CANDIDATE SET SIZE ===


,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,66218,5.3779,4.0,11.0,17.0,32.0,72


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== TWO-TOKEN ENTITY RECALL ===


,s1_with_true_matches,s1_all_matches_found,all_match_entity_recall_pct
0,2083574,17453.0,0.8376


In [14]:
print("Available candidate tables:")

display(
    con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_name ILIKE '%candidate%'
        ORDER BY table_name
    """).df()
)

Available candidate tables:


,table_name
0,s2_token2_candidates
1,s3_token2_candidates
2,token2_candidates


In [15]:
print("Creating exact-name candidates...")

con.execute("""
CREATE OR REPLACE TABLE s2_exact_candidates AS
SELECT
    s1.s1_id,
    s2.matched_id
FROM s1_data s1
JOIN s2_data s2
    ON s1.country = s2.country
   AND s1.name_normalized = s2.name_normalized
""")

print("S2 exact candidates created.")

con.execute("""
CREATE OR REPLACE TABLE s3_exact_candidates AS
SELECT
    s1.s1_id,
    s3.matched_id
FROM s1_data s1
JOIN s3_data s3
    ON s1.country = s3.country
   AND s1.name_normalized = s3.name_normalized
""")

print("S3 exact candidates created.")

con.execute("""
CREATE OR REPLACE TABLE exact_name_candidates AS
SELECT s1_id, matched_id FROM s2_exact_candidates
UNION
SELECT s1_id, matched_id FROM s3_exact_candidates
""")

print("======================================")
print("Exact-name candidate generation done")
print("======================================")

Creating exact-name candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 exact candidates created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S3 exact candidates created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exact-name candidate generation done


In [16]:
print("Combining exact-name + two-token candidates...")

con.execute("""
CREATE OR REPLACE TABLE combined_candidates AS
SELECT s1_id, matched_id
FROM exact_name_candidates

UNION

SELECT s1_id, matched_id
FROM token2_candidates
""")

print("Combined candidate table created.")

# ---------------------------------------
# 1. Pair recall
# ---------------------------------------
combined_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(cc.matched_id) AS true_pairs_found,
    COUNT(cc.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN combined_candidates cc
    ON gt.s1_id = cc.s1_id
   AND gt.matched_id = cc.matched_id
""").df()

print("\n=== COMBINED PAIR RECALL ===")
display(combined_recall.round(4))


# ---------------------------------------
# 2. Candidate set size
# ---------------------------------------
combined_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM combined_candidates
    GROUP BY s1_id
)
SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== COMBINED CANDIDATE SET SIZE ===")
display(combined_stats.round(4))


# ---------------------------------------
# 3. Entity-level all-match recall
# ---------------------------------------
combined_entity = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),
found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(cc.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN combined_candidates cc
        ON gt.s1_id = cc.s1_id
       AND gt.matched_id = cc.matched_id
    GROUP BY gt.s1_id
)
SELECT
    COUNT(*) AS s1_with_true_matches,
    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,
    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct
FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== COMBINED ENTITY RECALL ===")
display(combined_entity.round(4))

Combining exact-name + two-token candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Combined candidate table created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== COMBINED PAIR RECALL ===


...


=== COMBINED CANDIDATE SET SIZE ===


,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,1194230,10.5904,2.0,28.0,50.0,156.0,785


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== COMBINED ENTITY RECALL ===


,s1_with_true_matches,s1_all_matches_found,all_match_entity_recall_pct
0,2083574,38336.0,1.8399


In [17]:
print("Re-checking exact-name candidates...")

check = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(ec.matched_id) AS true_pairs_found,
    COUNT(ec.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN exact_name_candidates ec
    ON gt.s1_id = ec.s1_id
   AND gt.matched_id = ec.matched_id
""").df()

display(check.round(4))

Re-checking exact-name candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_true_pairs,true_pairs_found,pair_recall_pct
0,7638365,821025,10.7487


In [18]:
print("=== EXACT CANDIDATE DIAGNOSTIC ===")

# 1. Candidate row counts
counts = con.execute("""
SELECT
    'S2' AS source,
    COUNT(*) AS candidate_rows,
    COUNT(DISTINCT s1_id) AS unique_s1
FROM s2_exact_candidates

UNION ALL

SELECT
    'S3' AS source,
    COUNT(*) AS candidate_rows,
    COUNT(DISTINCT s1_id) AS unique_s1
FROM s3_exact_candidates

UNION ALL

SELECT
    'COMBINED' AS source,
    COUNT(*) AS candidate_rows,
    COUNT(DISTINCT s1_id) AS unique_s1
FROM exact_name_candidates
""").df()

display(counts)


# 2. Check source-table sizes
print("\n=== SOURCE TABLE SIZES ===")

display(
    con.execute("""
    SELECT 'S1' AS source, COUNT(*) AS rows FROM s1_data
    UNION ALL
    SELECT 'S2', COUNT(*) FROM s2_data
    UNION ALL
    SELECT 'S3', COUNT(*) FROM s3_data
    """).df()
)


# 3. Directly calculate exact-name matches again
print("\n=== DIRECT EXACT-NAME MATCH COUNT ===")

direct = con.execute("""
SELECT COUNT(*) AS direct_candidate_pairs
FROM s1_data s1
JOIN s2_data s2
    ON s1.country = s2.country
   AND s1.name_normalized = s2.name_normalized

UNION ALL

SELECT COUNT(*) AS direct_candidate_pairs
FROM s1_data s1
JOIN s3_data s3
    ON s1.country = s3.country
   AND s1.name_normalized = s3.name_normalized
""").df()

display(direct)


# 4. Check GT source distribution
print("\n=== GT MATCH SOURCE DISTRIBUTION ===")

display(
    con.execute("""
    SELECT
        CASE
            WHEN matched_id LIKE 'S2-%' THEN 'S2'
            WHEN matched_id LIKE 'S3-%' THEN 'S3'
            ELSE 'OTHER'
        END AS source,
        COUNT(*) AS true_matches
    FROM gt_pairs
    GROUP BY 1
    ORDER BY 1
    """).df()
)

=== EXACT CANDIDATE DIAGNOSTIC ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source,candidate_rows,unique_s1
0,S2,6020803,835014
1,S3,6301151,870087
2,COMBINED,12321954,1150744



=== SOURCE TABLE SIZES ===


,source,rows
0,S1,2206821
1,S2,5034616
2,S3,5285603



=== DIRECT EXACT-NAME MATCH COUNT ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,direct_candidate_pairs
0,6020803
1,6301151



=== GT MATCH SOURCE DISTRIBUTION ===


,source,true_matches
0,S2,3693619
1,S3,3944746


In [19]:
print("=== EXACT-NAME RECALL BY SOURCE ===")

source_recall = con.execute("""
SELECT
    CASE
        WHEN gt.matched_id LIKE 'S2-%' THEN 'S2'
        WHEN gt.matched_id LIKE 'S3-%' THEN 'S3'
    END AS source,

    COUNT(*) AS total_true_pairs,

    COUNT(ec.matched_id) AS true_pairs_found,

    COUNT(ec.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct

FROM gt_pairs gt

LEFT JOIN exact_name_candidates ec
    ON gt.s1_id = ec.s1_id
   AND gt.matched_id = ec.matched_id

GROUP BY 1
ORDER BY 1
""").df()

display(source_recall.round(4))

=== EXACT-NAME RECALL BY SOURCE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source,total_true_pairs,true_pairs_found,pair_recall_pct
0,S2,3693619,407332,11.0280
1,S3,3944746,413693,10.4872


In [20]:
print("Creating rare-token candidates (frequency <= 100)...")

con.execute("""
CREATE OR REPLACE TABLE s2_rare_candidates AS
SELECT DISTINCT
    s1.s1_id,
    s2.matched_id
FROM s1_rare_tokens s1
JOIN s2_rare_tokens s2
    ON s1.country = s2.country
   AND s1.token = s2.token
""")

print("S2 rare-token candidates created.")

con.execute("""
CREATE OR REPLACE TABLE s3_rare_candidates AS
SELECT DISTINCT
    s1.s1_id,
    s3.matched_id
FROM s1_rare_tokens s1
JOIN s3_rare_tokens s3
    ON s1.country = s3.country
   AND s1.token = s3.token
""")

print("S3 rare-token candidates created.")

con.execute("""
CREATE OR REPLACE TABLE rare_token_candidates AS
SELECT s1_id, matched_id
FROM s2_rare_candidates

UNION

SELECT s1_id, matched_id
FROM s3_rare_candidates
""")

print("======================================")
print("Rare-token candidate generation done")
print("======================================")

Creating rare-token candidates (frequency <= 100)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 rare-token candidates created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S3 rare-token candidates created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rare-token candidate generation done


In [21]:
print("Evaluating rare-token blocker...")

# =======================================
# 1. Pair recall
# =======================================
rare_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_pairs,
    COUNT(rc.matched_id) AS true_pairs_found,
    COUNT(rc.matched_id) * 100.0 / COUNT(*) AS pair_recall_pct
FROM gt_pairs gt
LEFT JOIN rare_token_candidates rc
    ON gt.s1_id = rc.s1_id
   AND gt.matched_id = rc.matched_id
""").df()

print("\n=== RARE-TOKEN PAIR RECALL ===")
display(rare_recall.round(4))


# =======================================
# 2. Candidate set size
# =======================================
rare_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM rare_token_candidates
    GROUP BY s1_id
)
SELECT
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n=== RARE-TOKEN CANDIDATE SET SIZE ===")
display(rare_stats.round(4))


# =======================================
# 3. Entity-level recall
# =======================================
rare_entity = con.execute("""
WITH gt_counts AS (
    SELECT
        s1_id,
        COUNT(*) AS true_count
    FROM gt_pairs
    GROUP BY s1_id
),
found_counts AS (
    SELECT
        gt.s1_id,
        COUNT(rc.matched_id) AS found_count
    FROM gt_pairs gt
    LEFT JOIN rare_token_candidates rc
        ON gt.s1_id = rc.s1_id
       AND gt.matched_id = rc.matched_id
    GROUP BY gt.s1_id
)
SELECT
    COUNT(*) AS s1_with_true_matches,
    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) AS s1_all_matches_found,
    SUM(
        CASE
            WHEN found_count = true_count THEN 1
            ELSE 0
        END
    ) * 100.0 / COUNT(*) AS all_match_entity_recall_pct
FROM gt_counts
JOIN found_counts USING (s1_id)
""").df()

print("\n=== RARE-TOKEN ENTITY RECALL ===")
display(rare_entity.round(4))

Evaluating rare-token blocker...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== RARE-TOKEN PAIR RECALL ===


,total_true_pairs,true_pairs_found,pair_recall_pct
0,7638365,1697799,22.2273


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== RARE-TOKEN CANDIDATE SET SIZE ===


,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,594483,46.052,39.0,92.0,98.0,148.0,282


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== RARE-TOKEN ENTITY RECALL ===


,s1_with_true_matches,s1_all_matches_found,all_match_entity_recall_pct
0,2083574,306711.0,14.7204


In [22]:
print("🚀 Creating FINAL candidate set...")

# ==========================================
# FINAL CANDIDATE SET
# ==========================================
con.execute("""
CREATE OR REPLACE TABLE final_candidates AS

SELECT s1_id, matched_id
FROM exact_name_candidates

UNION

SELECT s1_id, matched_id
FROM token2_candidates

UNION

SELECT s1_id, matched_id
FROM rare_token_candidates
""")

print("✅ Final candidate table created.")


# ==========================================
# FINAL CANDIDATE STATISTICS
# ==========================================
final_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM final_candidates
    GROUP BY s1_id
)
SELECT
    COUNT(*) AS total_candidate_pairs,
    COUNT(*) AS unique_candidate_rows,
    COUNT(DISTINCT s1_id) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

print("\n========== FINAL CANDIDATE STATS ==========")
display(final_stats.round(4))


# ==========================================
# CREATE M3 SAMPLE
# ==========================================
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/business_entity_resolution/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample_path = OUTPUT_DIR / "candidate_pairs_sample_1000.tsv"

con.execute(f"""
COPY (
    SELECT
        s1_id AS source1_entity_id,
        matched_id AS matched_entity_id
    FROM final_candidates
    USING SAMPLE 1000
)
TO '{sample_path}'
WITH (HEADER, DELIMITER '\t')
""")

print("\n✅ M3 sample created:")
print(sample_path)


# ==========================================
# SHOW SAMPLE
# ==========================================
display(
    con.execute("""
        SELECT
            s1_id AS source1_entity_id,
            matched_id AS matched_entity_id
        FROM final_candidates
        LIMIT 10
    """).df()
)

🚀 Creating FINAL candidate set...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Final candidate table created.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


========== FINAL CANDIDATE STATS ==========


,total_candidate_pairs,unique_candidate_rows,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,1529812,1529812,1529812,25.7246,8.0,78.0,94.0,158.0,785



✅ M3 sample created:
/content/drive/MyDrive/business_entity_resolution/output/candidate_pairs_sample_1000.tsv


,source1_entity_id,matched_entity_id
0,S1-996483571,S2-201046344
1,S1-994731416,S2-412234296
2,S1-31297272,S2-700528474
3,S1-862943649,S2-592084228
4,S1-146453713,S2-941970903
5,S1-626257975,S2-807511740
6,S1-873922292,S2-822101519
7,S1-845374931,S2-88505174
8,S1-86790189,S2-92274089
9,S1-540131352,S2-416631515


In [23]:
final_stats = con.execute("""
WITH counts AS (
    SELECT
        s1_id,
        COUNT(*) AS candidate_count
    FROM final_candidates
    GROUP BY s1_id
)
SELECT
    (SELECT COUNT(*) FROM final_candidates) AS total_candidate_pairs,
    COUNT(*) AS unique_s1_with_candidates,
    AVG(candidate_count) AS avg_candidates,
    quantile_cont(candidate_count, 0.50) AS p50_candidates,
    quantile_cont(candidate_count, 0.90) AS p90_candidates,
    quantile_cont(candidate_count, 0.95) AS p95_candidates,
    quantile_cont(candidate_count, 0.99) AS p99_candidates,
    MAX(candidate_count) AS max_candidates
FROM counts
""").df()

display(final_stats.round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_candidate_pairs,unique_s1_with_candidates,avg_candidates,p50_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,39353766,1529812,25.7246,8.0,78.0,94.0,158.0,785


In [24]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/business_entity_resolution/output/candidate_pairs_sample_1000.tsv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>